### Calculate the number of parameters and FLOPs

In [ ]:
import torch
from torch.utils.data import DataLoader

from omegaconf import OmegaConf
from models.vqvae_arch import VQVAEGANMultiHeadTransformer

from fvcore.nn import FlopCountAnalysis, parameter_count_table

config = OmegaConf.load(r"configs/SMILE_CR/VQGAN.yaml")
model = VQVAEGANMultiHeadTransformer(**config.model.params.ddconfig.params)
ckpt = torch.load(r"your_ckpt_path.ckpt")["state_dict"]
new_ckpt = {k[6:]: v for k, v in ckpt.items() if k.startswith("vqvae.")}; del ckpt
model.load_state_dict(new_ckpt, strict=True)
model.cuda()
model.eval()

print(parameter_count_table(model))
flops = FlopCountAnalysis(model, torch.randn(1, 14, 256, 256).cuda()).total()
print(f"FLOPs: {flops / 1e9:.2f} GFLOPs")


### Compute metrics

In [ ]:
from data.datasets import SMILE_CR
from metric import calculate_metrics

data_ds = SMILE_CR(r"C:\Users\Administrator\.data\SMILE-CR\TestData")
data_iter = DataLoader(data_ds, batch_size=1, shuffle=False, num_workers=0)

with torch.no_grad():
    avg_ssim, avg_psnr, avg_mae, avg_sam = calculate_metrics(model, data_iter, "cuda", False)
    
print(avg_ssim, avg_psnr, avg_mae, avg_sam)